In [37]:
# Зареждане на необходимите библиотеки
import torch
import torch.nn.functional as F

from torch_geometric.datasets import Planetoid

In [ ]:
# Зареждане от файл
graph = torch.load("patients_graph.pt")

from torch_geometric.data import Data

data = Data(
    x=graph["x"],
    edge_index=graph["edge_index"],
    y=graph["y"]
)
print(data)

In [2]:
# Представяне на графа
from torch_geometric.data import Data

# 3 върха, по 2 характеристики
x = torch.tensor([
    [1.0, 0.5],
    [0.3, 2.1],
    [1.2, 1.8]
], dtype=torch.float)

# Ребра: 0→1, 1→0, 1→2, 2→1
edge_index = torch.tensor([
    [0, 1, 1, 2],
    [1, 0, 2, 1]
], dtype=torch.long)

# Един клас за целия граф
y = torch.tensor([1])

data = Data(x=x, edge_index=edge_index, y=y)

print(data)

Data(x=[3, 2], edge_index=[2, 4], y=[1])


In [19]:
# Зареждане от файл
import torch
graph = torch.load("patients_graph.pt")

from torch_geometric.data import Data

data = Data(x=graph["x"], edge_index=graph["edge_index"], y=graph["y"])
print(data)

Data(x=[5, 4], edge_index=[2, 4], y=[5])


In [20]:
# Разделяне на набор от данни за обучение и за тестване
num_nodes = data.num_nodes
perm = torch.randperm(num_nodes)
train_size = int(0.8 * num_nodes)

# Train mask
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[perm[:train_size]] = True # първите 80% от случайно разбърканите индекси.
# Test mask
test_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask[perm[train_size:]] = True # останалите 20%.

data.train_mask = train_mask
data.test_mask = test_mask

In [21]:
# Дефиниране на GCN модел
from torch_geometric.nn import GCNConv

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()

        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)

        return x


In [30]:
# Създаване на модела
model = GCN(
    in_channels=4,
    hidden_channels=3,
    out_channels=2)


In [32]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4)

criterion = torch.nn.CrossEntropyLoss()


In [33]:
# Обучение на модела
for epoch in range(200):
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(
        out[data.train_mask],
        data.y[data.train_mask])
    loss.backward()
    optimizer.step()


In [34]:
# Класифициране
pred = out.argmax(dim=1)

In [35]:
# Оценяване на модела
correct = (pred[data.test_mask] == data.y[data.test_mask])
accuracy = int(correct.sum()) / int(data.test_mask.sum())
print(accuracy)


0.806


In [36]:
# Пресмятане на допълнителни мерки
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_true = data.y[data.test_mask].cpu()
y_pred = pred[data.test_mask].cpu()

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='macro')
recall = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

Accuracy : 0.8060
Precision: 0.7844
Recall   : 0.8208
F1-score : 0.7988


In [28]:
# Пример с Cora
# Зареждане на набора от данни
dataset = Planetoid(root='data/Cora', name='Cora')
data = dataset[0]

print(data)

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


In [31]:
# За Cora създаването на модела трябва да е при следните параметри
model = GCN(
    in_channels=1433,
    hidden_channels=16,
    out_channels=7)

# Следва дефиниране на оптимизатор